# WalkThru — YOUR ROOM VIDEO → walkable 3D (Colab, T4 GPU)

Point this at a phone video of a room in your Google Drive; get back a GLB you can walk a character through. Every mistake we hit is already fixed here.

## Do this first
1. **Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save**
2. Put your video in Google Drive at **`MyDrive/WalkThru/`** (any name, .mp4/.mov). If it's the only video there, the notebook finds it automatically.
3. Run the cells **top to bottom**. Each ends with a ✅ line. If one fails, copy its whole output and send it back.

⚠️ **CELL 2 restarts the runtime** — the 'session crashed' popup is NORMAL. After it, continue from CELL 3 (do not rerun 1–2).

### How to film a room so this works (if your first try looks bad, re-film like this)
- **Keep walking the whole time — never spin on the spot.** Motion between frames is what creates 3D.
- Move slowly and smoothly; do a full loop and end where you started.
- Good even light; avoid mirrors, glass, and big blank walls as the main subject.
- 1080p is plenty. 60–120 seconds is ideal.

In [ ]:
# CELL 1 — confirm the GPU is on
!nvidia-smi | head -12
print('✅ CELL 1 — you should see a Tesla T4 above. If not: Runtime ▸ Change runtime type ▸ T4 GPU.')

In [ ]:
# CELL 2 — install conda.  RUNTIME RESTARTS ('session crashed' popup = NORMAL).
# After the restart, continue from CELL 3. Do NOT rerun this cell.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CELL 3 — install COLMAP (CUDA build). ~3-5 min.
import os, shutil, subprocess
if shutil.which('mamba') is None:
    raise RuntimeError('mamba missing -> CELL 2 has not finished in THIS runtime. Run CELL 2, wait for the restart, then rerun CELL 3.')
# LESSON: condacolab writes a python pin that contradicts what it installed,
# which makes mamba refuse ALL installs. Remove it first.
pin = '/usr/local/conda-meta/pinned'
if os.path.exists(pin):
    print('removing bogus pin:', open(pin).read().strip())
    os.remove(pin)
# LESSON: the conda colmap package does NOT pull these runtime libs on Colab;
# without them colmap dies with 'cannot open shared object file'. Name them.
r = subprocess.run(['mamba','install','-y','-q','-c','conda-forge','colmap','libfaiss','openimageio'], capture_output=True, text=True)
print(r.stdout[-800:]); print(r.stderr[-500:])
assert r.returncode == 0, 'mamba install FAILED — send this whole output'
v = subprocess.run(['colmap','-h'], capture_output=True, text=True)
out = (v.stdout or '') + (v.stderr or '')
print('\n'.join(out.splitlines()[:6]))
if 'cannot open shared object' in out:
    print(subprocess.run(['bash','-lc',"ldd /usr/local/bin/colmap | grep 'not found'"], capture_output=True, text=True).stdout)
assert 'COLMAP' in out and 'cannot open shared object' not in out, 'colmap does not run — send this output'
print('✅ CELL 3 — colmap runs (header should say \'with CUDA\')')

In [ ]:
# CELL 4 — find the video in Drive, extract SHARP frames (drops motion-blurred ones)
RUN_NAME      = 'myroom'   # names the Drive output folder for this scan
VIDEO         = ''         # leave '' to auto-find the only video in MyDrive/WalkThru/, or set a full path
TARGET_FRAMES = 100        # 60-120 is the sweet spot for one room

from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess, numpy as np
from PIL import Image

WORK = '/content/recon'; IMAGES = f'{WORK}/images'; RAW = f'{WORK}/raw_frames'
DRIVE_RUN = f'/content/drive/MyDrive/WalkThru/runs/{RUN_NAME}'
for d in (IMAGES, RAW, DRIVE_RUN): os.makedirs(d, exist_ok=True)

if not VIDEO:
    vids = []
    for ext in ('mp4','mov','MOV','MP4','m4v','avi'):
        vids += glob.glob(f'/content/drive/MyDrive/WalkThru/*.{ext}')
    assert vids, 'No video found in MyDrive/WalkThru/ — upload one, or set VIDEO to its full path'
    VIDEO = max(vids, key=os.path.getsize)
print('video:', VIDEO)

# duration -> extract ~1.7x target frames (we then throw away the blurry ones)
dur = float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',VIDEO], capture_output=True, text=True).stdout.strip())
fps = max(0.3, (TARGET_FRAMES * 1.7) / dur)
print(f'duration {dur:.0f}s -> extracting at {fps:.2f} fps (cap 1600px wide)')
for f in glob.glob(f'{RAW}/*.jpg'): os.remove(f)
r = subprocess.run(['ffmpeg','-y','-i',VIDEO,'-vf',f"fps={fps:.4f},scale='min(1600,iw)':-2",'-q:v','2',f'{RAW}/f_%04d.jpg'], capture_output=True, text=True)
assert r.returncode == 0, 'ffmpeg failed:\n' + r.stderr[-800:]
raw = sorted(glob.glob(f'{RAW}/*.jpg'))
print('raw frames:', len(raw))

# LESSON (real video): motion blur wrecks reconstruction. Score sharpness with a
# numpy Laplacian (no cv2 dependency) and drop the clearly-blurry frames.
def sharpness(path):
    g = np.asarray(Image.open(path).convert('L'), dtype=np.float64)
    lap = g[:-2,1:-1] + g[2:,1:-1] + g[1:-1,:-2] + g[1:-1,2:] - 4*g[1:-1,1:-1]
    return lap.var()
scores = np.array([sharpness(p) for p in raw])
keep_thr = 0.5 * np.median(scores)   # drop frames below half the median sharpness
keep = [p for p, s in zip(raw, scores) if s >= keep_thr]
# if still too many, evenly subsample down to ~1.2x target (keeps temporal spread)
cap = int(TARGET_FRAMES * 1.2)
if len(keep) > cap:
    idx = np.linspace(0, len(keep)-1, cap).round().astype(int)
    keep = [keep[i] for i in idx]
for f in glob.glob(f'{IMAGES}/*.jpg'): os.remove(f)
for i, p in enumerate(keep):
    os.replace(p, f'{IMAGES}/img_{i:04d}.jpg')
n = len(glob.glob(f'{IMAGES}/*.jpg'))
print(f'kept {n} sharp frames (dropped {len(raw)-n} blurry/extra)')
assert 30 <= n <= 250, f'{n} frames — aim ~60-120. Adjust TARGET_FRAMES or film a longer, slower pass.'
print('✅ CELL 4 done')

In [ ]:
# CELL 5 — features + matching on the GPU (~2-6 min)
import subprocess, time, shutil
assert shutil.which('colmap'), 'colmap missing -> runtime was recycled. Rerun CELLS 2,3,4 in order.'
DB = f'{WORK}/db.db'
if os.path.exists(DB): os.remove(DB)

def run(name, args, allow_fail=False):
    t = time.time(); print(f'=== {name} ===', flush=True)
    r = subprocess.run(args, capture_output=True, text=True)
    print(((r.stdout or '')[-900:] + '\n' + (r.stderr or '')[-900:]).strip())
    if r.returncode != 0 and not allow_fail: raise RuntimeError(f'{name} FAILED exit {r.returncode}')
    print(f'--- {name}: {time.time()-t:.0f}s (exit {r.returncode})'); return r.returncode

def flag(cmd, new, old):
    h = subprocess.run(['colmap',cmd,'--help'], capture_output=True, text=True)
    return new if new.lstrip('-') in ((h.stdout or '')+(h.stderr or '')) else old
EX_GPU = flag('feature_extractor','--FeatureExtraction.use_gpu','--SiftExtraction.use_gpu')
MA_GPU = flag('exhaustive_matcher','--FeatureMatching.use_gpu','--SiftMatching.use_gpu')

# LESSON: a real phone video has NO known intrinsics. Let COLMAP self-calibrate
# with the OPENCV model + single-camera. (Do NOT invent an intrinsics.json — a
# wrong focal length silently ruins the whole solve.)
run('extract features (GPU)', ['colmap','feature_extractor','--database_path',DB,'--image_path',IMAGES,
    '--ImageReader.single_camera','1','--ImageReader.camera_model','OPENCV',EX_GPU,'1'])

n = len(glob.glob(f'{IMAGES}/*.jpg'))
# LESSON: exhaustive matching is the most robust (it finds loop closures for
# free) and is cheap on GPU for a room-sized set. Use sequential only if huge.
if n <= 160:
    run('match: exhaustive (GPU)', ['colmap','exhaustive_matcher','--database_path',DB,MA_GPU,'1'])
else:
    run('match: sequential (GPU)', ['colmap','sequential_matcher','--database_path',DB,'--SequentialMatching.overlap','15',MA_GPU,'1'])
print('✅ CELL 5 done')

In [ ]:
# CELL 6 — solve camera poses (SfM, CPU ~3-15 min) + checkpoint to Drive
SPARSE = f'{WORK}/sparse'; os.makedirs(SPARSE, exist_ok=True)
margs = ['colmap','mapper','--database_path',DB,'--image_path',IMAGES,'--output_path',SPARSE]
if not os.path.isdir(f'{SPARSE}/0'):
    rc = run('SfM', margs, allow_fail=True)
    # LESSON: textureless rooms sometimes fail the default bootstrap -> relaxed retry
    if rc != 0 or not os.path.isdir(f'{SPARSE}/0'):
        run('SfM retry (relaxed init)', margs + ['--Mapper.init_min_num_inliers','50','--Mapper.init_min_tri_angle','4'])
else:
    print('sparse/0 exists — skipping (resumable)')
run('model report', ['colmap','model_analyzer','--path',f'{SPARSE}/0'])
# checkpoint images+sparse+db to Drive (survives a runtime recycle; also the
# input for the Gaussian-splat notebook)
subprocess.run(['bash','-lc',f"cp -r {SPARSE} '{DRIVE_RUN}/' && cp -r {IMAGES} '{DRIVE_RUN}/' && cp {DB} '{DRIVE_RUN}/'"], check=True)
print('checkpointed to', DRIVE_RUN)
print('✅ CELL 6 — SEND ME the report: Registered images (want most of your frames) + Mean reprojection error (<1px is great)')

In [ ]:
# CELL 7 — undistort images (seconds, resumable)
DENSE = f'{WORK}/dense'
if not os.path.isdir(f'{DENSE}/images'):
    run('undistort', ['colmap','image_undistorter','--image_path',IMAGES,'--input_path',f'{SPARSE}/0','--output_path',DENSE,'--output_type','COLMAP'])
else:
    print('dense workspace exists — skipping')
print('✅ CELL 7 done')

In [ ]:
# CELL 8 — GPU dense depth. draft ≈ 10-15 min, full ≈ 40-70 min (T4).
QUALITY = 'full'   # use 'draft' the first time to check the capture is good, then 'full'
opts = {'draft': ['--PatchMatchStereo.max_image_size','1000','--PatchMatchStereo.geom_consistency','false'],
        'full':  ['--PatchMatchStereo.max_image_size','1600','--PatchMatchStereo.geom_consistency','true']}[QUALITY]
run('patch match stereo (GPU)', ['colmap','patch_match_stereo','--workspace_path',DENSE,'--workspace_format','COLMAP',
    '--PatchMatchStereo.gpu_index','0','--PatchMatchStereo.cache_size','12',*opts])
print('✅ CELL 8 done')

In [ ]:
# CELL 9 — fuse depth into a colored point cloud (~4-8 min) + checkpoint
ftype = 'geometric' if QUALITY == 'full' else 'photometric'
run('stereo fusion', ['colmap','stereo_fusion','--workspace_path',DENSE,'--workspace_format','COLMAP',
    '--input_type',ftype,'--output_path',f'{DENSE}/fused.ply','--StereoFusion.cache_size','12'])
print(subprocess.run(['bash','-lc',f'ls -lh {DENSE}/fused.ply'], capture_output=True, text=True).stdout)
subprocess.run(['bash','-lc',f"cp {DENSE}/fused.ply '{DRIVE_RUN}/'"], check=True)
print('✅ CELL 9 — fused.ply checkpointed to Drive')

In [ ]:
# CELL 10 — Poisson mesh.  LESSON: NEVER pass --PoissonMeshing.trim (its
# surface-trimmer CRASHES in this build). We crop the extra hull in CELL 11.
if not os.path.exists(f'{DENSE}/mesh.ply'):
    run('poisson meshing', ['colmap','poisson_mesher','--input_path',f'{DENSE}/fused.ply','--output_path',f'{DENSE}/mesh.ply'])
else:
    print('mesh.ply exists — skipping')
print(subprocess.run(['bash','-lc',f'ls -lh {DENSE}/mesh.ply'], capture_output=True, text=True).stdout)
print('✅ CELL 10 done')

In [ ]:
# CELL 11 — clean + flip + export GLB.  LESSON: use trimesh, NOT open3d
# (open3d cannot import on the condacolab kernel).
!pip install -q trimesh scipy
import numpy as np, trimesh, gc
m = trimesh.load(f'{DENSE}/mesh.ply', process=False)
print(f'raw: {len(m.vertices):,} verts {len(m.faces):,} faces')
# crop the Poisson 'bowl' hull: keep faces inside the dense-region percentile box
lo = np.percentile(m.vertices, 2, axis=0); hi = np.percentile(m.vertices, 98, axis=0)
pad = 0.15 * (hi - lo); lo, hi = lo - pad, hi + pad
cent = m.vertices[m.faces].mean(axis=1)
m.update_faces(np.all((cent >= lo) & (cent <= hi), axis=1)); m.remove_unreferenced_vertices()
# keep the largest connected piece (drops floating junk)
lbl = trimesh.graph.connected_component_labels(m.face_adjacency, node_count=len(m.faces))
cnt = np.bincount(lbl)
if len(cnt) > 1 and cnt.max() > 0.5*len(m.faces):
    m.update_faces(lbl == cnt.argmax()); m.remove_unreferenced_vertices()
print(f'cleaned: {len(m.vertices):,} verts {len(m.faces):,} faces')
# LESSON: COLMAP's world is Y-DOWN. Flip 180° about X so the house is upright
# in the viewer (without this the character stands on the sky-side of the floor).
m.apply_transform(trimesh.transformations.rotation_matrix(np.pi, [1,0,0]))
GLB = f'/content/{RUN_NAME}_mesh.glb'
m.export(GLB); del m; gc.collect()
print(subprocess.run(['bash','-lc',f"ls -lh {GLB} && cp {GLB} '{DRIVE_RUN}/'"], capture_output=True, text=True).stdout)
print('✅ CELL 11 — GLB built and saved to Drive runs folder')

In [ ]:
# CELL 12 — download the GLB (also in Drive: WalkThru/runs/<RUN_NAME>/)
from google.colab import files
files.download(GLB)
print('✅ CELL 12 — done! Walk it on your laptop (see the notes below).')

## After download — on your laptop (Windows, `D:\startupidea`)

The GLB is full-resolution (can be hundreds of MB). Compress it for the web (we've seen 478 MB → 0.4 MB):
```
node node_modules\\@gltf-transform\\cli\\bin\\cli.js optimize <downloaded>.glb public\\scans\\myroom.glb --compress draco --simplify true --simplify-error 0.0001
```
Then walk through YOUR room:
```
npm run dev
# open  http://localhost:5173/?model=/scans/myroom.glb
```

### Photoreal upgrade (Gaussian splat)
Open **`WalkThru_Colab_Splats.ipynb`** in a **fresh runtime** (Runtime ▸ Disconnect and delete runtime first). It reuses THIS run's Drive checkpoint (`WalkThru/runs/<RUN_NAME>/`) — same frames, same camera solve — and trains a photoreal splat. Then in the viewer:
```
http://localhost:5173/?splat=/scans/myroom_splat.ply&collision=/scans/myroom.glb
```
The splat is what the buyer sees; this mesh is what the character walks on.

### If the result looks bad
- **Few 'Registered images' in CELL 6** → the video had too much rotation-in-place or motion blur. Re-film walking slowly and steadily; keep TARGET_FRAMES ~100.
- **Holes / thin walls** → film more overlap and a second loop at a different height.
- **Warped/melted look** → normal for mesh photogrammetry; the Gaussian-splat path fixes visual quality.